In [1]:
import os
import sys
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score

In [3]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("Torch version:", torch.__version__)

CUDA available: True
CUDA device name: NVIDIA GeForce RTX 4080 SUPER
CUDA version: 12.1
Torch version: 2.5.1+cu121


In [3]:
#!pip install mmengine

In [4]:
#!pip install lightning-template

In [2]:
sys.path.append(os.path.abspath('../'))  # Adjust this path based on where your files are located
print(sys.path)

from datasets.hint.hint_admet import HINTDataset 
from datasets.ctod.ctod_enrollment import CTODataset
from mmcto.mmcto import MMCTO
from mmcto.layers.sparse_moe import SparseMOELayer, FeedForwardLayer
from mmf.early_fusion import EarlyFusion
from mmf.middle_fusion import MiddleFusion
from mmf.late_fusion import LateFusion
from torch.utils.data import Dataset
import pytorch_lightning.callbacks as pl_callbacks

['C:\\Users\\Carol\\Documents\\Data Code and Deliverables\\Data Group Part\\project\\models', 'C:\\Users\\Carol\\anaconda3\\envs\\newthesis\\python311.zip', 'C:\\Users\\Carol\\anaconda3\\envs\\newthesis\\DLLs', 'C:\\Users\\Carol\\anaconda3\\envs\\newthesis\\Lib', 'C:\\Users\\Carol\\anaconda3\\envs\\newthesis', '', 'C:\\Users\\Carol\\anaconda3\\envs\\newthesis\\Lib\\site-packages', 'C:\\Users\\Carol\\anaconda3\\envs\\newthesis\\Lib\\site-packages\\win32', 'C:\\Users\\Carol\\anaconda3\\envs\\newthesis\\Lib\\site-packages\\win32\\lib', 'C:\\Users\\Carol\\anaconda3\\envs\\newthesis\\Lib\\site-packages\\Pythonwin', 'C:\\Users\\Carol\\Documents\\Data Code and Deliverables\\Data Group Part\\project']


In [3]:
PHASE = "II"
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 2e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_DIM = 768

In [4]:
base_path = r"C:\Users\Carol\Documents\Data Code and Deliverables\Data Group Part\Data\clinical-trial-outcome-prediction\Processed\hint"
#base_path = r"C:/Users/Asus/Documents/Data and Code Deliverables/Data Group Part/Data/clinical-trial-outcome-prediction/Processed/ctod"
#admet_path = os.path.join(base_path, "Admet", "cooked")

data_prefix = dict(
    data_path=base_path,
    table_path=os.path.join(base_path, "text_description"),
    summarization_path=os.path.join(base_path, "brief_summary"),
    drug_description_path=os.path.join(base_path, "drugbank", "druginfo_description.json"),
    criteria_path=os.path.join(base_path, "criteria"),
    #admet_path=admet_path,
)

In [5]:
def move_to_device(obj, device):
    if torch.is_tensor(obj):
        return obj.to(device)
    elif isinstance(obj, dict):
        return {k: move_to_device(v, device) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [move_to_device(v, device) for v in obj]
    return obj

In [6]:
def build_encoder():
    return nn.TransformerEncoder(
        nn.TransformerEncoderLayer(
            d_model=MODEL_DIM,
            nhead=8,
            dim_feedforward=2048,
            dropout=0.1,
            activation='relu',
            batch_first=True
        ),
        num_layers=2
    )
input_parts = [
    "table", "summarization", "description", "criteria",
    "smiles", "smiles_concat", "smiles_summarization",
    "drugs", "drugs_concat", "drugs_summarization",
    "diseases", "diseases_concat", "diseases_summarization",
    "smiles_transformer_concat", "enrollment"
]

encoders = nn.ModuleDict({k: build_encoder() for k in input_parts})
smoe_encoder = SparseMOELayer(
    expert_cfg=FeedForwardLayer,
    num_experts=4,
    input_dim=MODEL_DIM,
    topk=2,
)

In [7]:
model = MMCTO(
    encoders=encoders,
    smoe_encoder=smoe_encoder,
    aux_loss=True,
    aux_loss_share_fc=False,
    moe_method="weighted",
    vocab_size=28996,
    model_dim=MODEL_DIM,
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [8]:
train_dataset = HINTDataset(
    data_prefix=data_prefix,
    ann_file_name=f"phase_{PHASE}_train",
    serialize_data=False,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=train_dataset.collate_fn,
)

test_dataset = HINTDataset(
    data_prefix=data_prefix,
    ann_file_name=f"phase_{PHASE}_test",
    serialize_data=False,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=test_dataset.collate_fn,
)

[Cache] Saved: C:\Users\Carol\Documents\Data Code and Deliverables\Data Group Part\Data\clinical-trial-outcome-prediction\Processed\hint\phase_III_train_tokenized.pt
[Cache] Saved: C:\Users\Carol\Documents\Data Code and Deliverables\Data Group Part\Data\clinical-trial-outcome-prediction\Processed\hint\phase_III_test_tokenized.pt


In [9]:
# train_dataset = CTODataset(
#     data_prefix=data_prefix,
#     ann_file_name=f"phase_{PHASE}_train",
#     serialize_data=False,
# )
# train_loader = DataLoader(
#     train_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=True,
#     collate_fn=train_dataset.collate_fn,
# )

# test_dataset = CTODataset(
#     data_prefix=data_prefix,
#     ann_file_name=f"phase_{PHASE}_valid",
#     serialize_data=False,
# )
# test_loader = DataLoader(
#     test_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     collate_fn=test_dataset.collate_fn,
# )

[Cache] Saved: C:\Users\Carol\Documents\Data Code and Deliverables\Data Group Part\Data\clinical-trial-outcome-prediction\Processed\ctod\phase_III_train_tokenized.pt
[Cache] Saved: C:\Users\Carol\Documents\Data Code and Deliverables\Data Group Part\Data\clinical-trial-outcome-prediction\Processed\ctod\phase_III_valid_tokenized.pt


In [9]:
param_device = next(model.parameters()).device
print(f"[INFO] Model is on device: {param_device}")

# Confirm GPU memory usage
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**2  # in MB
    reserved = torch.cuda.memory_reserved() / 1024**2    # in MB
    print(f"[INFO] GPU Memory - Allocated: {allocated:.2f} MB | Reserved: {reserved:.2f} MB")
else:
    print("[WARNING] CUDA is not available. Using CPU.")

[INFO] Model is on device: cuda:0
[INFO] GPU Memory - Allocated: 671.15 MB | Reserved: 722.00 MB


## HINT

### PHASE I

In [9]:
print(f"\nTraining MMCTO on Phase {PHASE}...\n")
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for batch in train_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)
        loss = out["loss_dict"]["loss"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")



Training MMCTO on Phase I...

Epoch 1 | Avg Loss: 0.4111
Epoch 2 | Avg Loss: 0.3920
Epoch 3 | Avg Loss: 0.3801
Epoch 4 | Avg Loss: 0.3781
Epoch 5 | Avg Loss: 0.3692
Epoch 6 | Avg Loss: 0.3512
Epoch 7 | Avg Loss: 0.3183
Epoch 8 | Avg Loss: 0.2658
Epoch 9 | Avg Loss: 0.2023
Epoch 10 | Avg Loss: 0.1455
Epoch 11 | Avg Loss: 0.1107
Epoch 12 | Avg Loss: 0.0572
Epoch 13 | Avg Loss: 0.0389
Epoch 14 | Avg Loss: 0.0289
Epoch 15 | Avg Loss: 0.0260
Epoch 16 | Avg Loss: 0.0361
Epoch 17 | Avg Loss: 0.0323
Epoch 18 | Avg Loss: 0.0211
Epoch 19 | Avg Loss: 0.0294
Epoch 20 | Avg Loss: 0.0155


In [10]:
torch.save(model.state_dict(), "mmcto_phaseI.pth")

In [11]:
print(f"\nEvaluating on Phase {PHASE} Test Set...\n")
model.eval()

all_preds = {"fused": [], "target": []}
modalities = model.final_input_parts
for mod in modalities:
    all_preds[mod] = []

with torch.no_grad():
    for batch in test_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)

        fused_preds = out["metric_dict"].get("preds")
        target = batch["label"]

        if fused_preds is not None:
            all_preds["fused"].append(fused_preds.cpu().numpy())
            all_preds["target"].append(target.cpu().numpy())

        for mod in modalities:
            mod_pred = out["metric_dict"].get(mod)
            if mod_pred is not None:
                all_preds[mod].append(mod_pred.cpu().numpy())

for key in all_preds:
    if all_preds[key]:
        all_preds[key] = np.concatenate(all_preds[key])
    else:
        all_preds[key] = np.array([])


Evaluating on Phase I Test Set...



C:\Users\Carol\anaconda3\envs\newthesis\Lib\site-packages\torch\nn\modules\transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(


In [12]:
def compute_metrics(preds, targets):
    preds_binary = (preds > 0.5).astype(int)
    pr = average_precision_score(targets, preds)
    roc = roc_auc_score(targets, preds)
    f1 = f1_score(targets, preds_binary)
    return pr * 100, f1 * 100, roc * 100

print(f"{'Modality':<25} {'PR':>6} {'F1':>6} {'ROC':>6}")
print("=" * 45)

for mod in modalities:
    if mod in all_preds and all_preds[mod].size:
        pr, f1, roc = compute_metrics(all_preds[mod], all_preds["target"])
        print(f"{mod:<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

pr, f1, roc = compute_metrics(all_preds["fused"], all_preds["target"])
print(f"{'All (Fused)':<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

Modality                      PR     F1    ROC
table                      71.03  32.17  38.84
summarization              85.39  87.59  67.78
smiles                     78.78   0.00  49.51
description                74.03   5.72  45.22
criteria                   75.14  24.97  45.35
enrollment                 77.49   0.00  50.00
diseases                   80.54  87.09  56.99
drugs                      78.87  64.39  52.61
All (Fused)                86.88  87.50  69.14


## PHASE II

In [10]:
print(f"\nTraining MMCTO on Phase {PHASE}...\n")
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for batch in train_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)
        loss = out["loss_dict"]["loss"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")



Training MMCTO on Phase II...

Epoch 1 | Avg Loss: 0.5061
Epoch 2 | Avg Loss: 0.5021
Epoch 3 | Avg Loss: 0.4983
Epoch 4 | Avg Loss: 0.4942
Epoch 5 | Avg Loss: 0.4750
Epoch 6 | Avg Loss: 0.4449
Epoch 7 | Avg Loss: 0.4067
Epoch 8 | Avg Loss: 0.3544
Epoch 9 | Avg Loss: 0.3061
Epoch 10 | Avg Loss: 0.2551
Epoch 11 | Avg Loss: 0.2200
Epoch 12 | Avg Loss: 0.1661
Epoch 13 | Avg Loss: 0.1300
Epoch 14 | Avg Loss: 0.1007
Epoch 15 | Avg Loss: 0.0812
Epoch 16 | Avg Loss: 0.0739
Epoch 17 | Avg Loss: 0.0610
Epoch 18 | Avg Loss: 0.0488
Epoch 19 | Avg Loss: 0.0470
Epoch 20 | Avg Loss: 0.0215


In [11]:
torch.save(model.state_dict(), "mmcto_phaseII.pth")

In [12]:
from tqdm import tqdm
print(f"\nEvaluating on Phase {PHASE} Test Set...\n")
model.eval()

all_preds = {"fused": [], "target": []}
modalities = model.final_input_parts
for mod in modalities:
    all_preds[mod] = []

with torch.no_grad():
    for batch in test_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)

        fused_preds = out["metric_dict"].get("preds")
        target = batch["label"]

        if fused_preds is not None:
            all_preds["fused"].append(fused_preds.cpu().numpy())
            all_preds["target"].append(target.cpu().numpy())

        for mod in modalities:
            mod_pred = out["metric_dict"].get(mod)
            if mod_pred is not None:
                all_preds[mod].append(mod_pred.cpu().numpy())

for key in all_preds:
    if all_preds[key]:
        all_preds[key] = np.concatenate(all_preds[key])
    else:
        all_preds[key] = np.array([])


Evaluating on Phase II Test Set...



C:\Users\Carol\anaconda3\envs\newthesis\Lib\site-packages\torch\nn\modules\transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(


In [13]:
def compute_metrics(preds, targets):
    preds_binary = (preds > 0.5).astype(int)
    pr = average_precision_score(targets, preds)
    roc = roc_auc_score(targets, preds)
    f1 = f1_score(targets, preds_binary)
    return pr * 100, f1 * 100, roc * 100

print(f"{'Modality':<25} {'PR':>6} {'F1':>6} {'ROC':>6}")
print("=" * 45)

for mod in modalities:
    if mod in all_preds and all_preds[mod].size:
        pr, f1, roc = compute_metrics(all_preds[mod], all_preds["target"])
        print(f"{mod:<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

pr, f1, roc = compute_metrics(all_preds["fused"], all_preds["target"])
print(f"{'All (Fused)':<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

Modality                      PR     F1    ROC
table                      60.31   4.24  48.33
summarization              62.38  56.46  50.08
smiles                     61.11   3.44  49.03
description                67.91  75.17  58.11
criteria                   62.57  70.10  50.30
enrollment                 61.59   0.00  50.00
diseases                   57.44  51.34  44.31
drugs                      62.06  32.29  49.69
All (Fused)                68.22  75.98  58.74


## PHASE III

In [9]:
print(f"\nTraining MMCTO on Phase {PHASE} with ADMET features...\n")
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for batch in train_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)
        loss = out["loss_dict"]["loss"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")


Training MMCTO on Phase III with ADMET features...

Epoch 1 | Avg Loss: 0.4036
Epoch 2 | Avg Loss: 0.3935
Epoch 3 | Avg Loss: 0.3853
Epoch 4 | Avg Loss: 0.3622
Epoch 5 | Avg Loss: 0.3180
Epoch 6 | Avg Loss: 0.2765
Epoch 7 | Avg Loss: 0.2234
Epoch 8 | Avg Loss: 0.1676
Epoch 9 | Avg Loss: 0.1261
Epoch 10 | Avg Loss: 0.1084
Epoch 11 | Avg Loss: 0.0828
Epoch 12 | Avg Loss: 0.0712
Epoch 13 | Avg Loss: 0.0624
Epoch 14 | Avg Loss: 0.0546
Epoch 15 | Avg Loss: 0.0317
Epoch 16 | Avg Loss: 0.0417
Epoch 17 | Avg Loss: 0.0468
Epoch 18 | Avg Loss: 0.0235
Epoch 19 | Avg Loss: 0.0274
Epoch 20 | Avg Loss: 0.0210


In [10]:
torch.save(model.state_dict(), "mmcto_phaseIII.pth")

In [11]:
print(f"\nEvaluating on Phase {PHASE} Test Set...\n")
model.eval()

all_preds = {"fused": [], "target": []}
modalities = model.final_input_parts
for mod in modalities:
    all_preds[mod] = []

with torch.no_grad():
    for batch in test_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)

        fused_preds = out["metric_dict"].get("preds")
        target = batch["label"]

        if fused_preds is not None:
            all_preds["fused"].append(fused_preds.cpu().numpy())
            all_preds["target"].append(target.cpu().numpy())

        for mod in modalities:
            mod_pred = out["metric_dict"].get(mod)
            if mod_pred is not None:
                all_preds[mod].append(mod_pred.cpu().numpy())

for key in all_preds:
    if all_preds[key]:
        all_preds[key] = np.concatenate(all_preds[key])
    else:
        all_preds[key] = np.array([])


Evaluating on Phase III Test Set...



C:\Users\Carol\anaconda3\envs\newthesis\Lib\site-packages\torch\nn\modules\transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(


In [12]:
def compute_metrics(preds, targets):
    preds_binary = (preds > 0.5).astype(int)
    pr = average_precision_score(targets, preds)
    roc = roc_auc_score(targets, preds)
    f1 = f1_score(targets, preds_binary)
    return pr * 100, f1 * 100, roc * 100

print(f"{'Modality':<25} {'PR':>6} {'F1':>6} {'ROC':>6}")
print("=" * 45)

for mod in modalities:
    if mod in all_preds and all_preds[mod].size:
        pr, f1, roc = compute_metrics(all_preds[mod], all_preds["target"])
        print(f"{mod:<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

pr, f1, roc = compute_metrics(all_preds["fused"], all_preds["target"])
print(f"{'All (Fused)':<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

Modality                      PR     F1    ROC
table                      74.66  20.00  52.02
summarization              75.52   0.42  53.81
smiles                     71.27  29.05  46.51
description                74.11  82.39  53.20
criteria                   75.04  76.20  52.83
enrollment                 72.56  84.10  50.00
diseases                   74.19   4.51  53.63
drugs                      77.04  74.66  57.34
All (Fused)                77.75  80.27  56.28


# CTOD

### PHASE I

In [9]:
print(f"\nTraining MMCTO on Phase {PHASE} with ADMET features...\n")
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for batch in train_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)
        loss = out["loss_dict"]["loss"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")


Training MMCTO on Phase I with ADMET features...

Epoch 1 | Avg Loss: 0.5073
Epoch 2 | Avg Loss: 0.5689
Epoch 3 | Avg Loss: 0.4654
Epoch 4 | Avg Loss: 0.4555
Epoch 5 | Avg Loss: 0.4359
Epoch 6 | Avg Loss: 0.4047
Epoch 7 | Avg Loss: 0.3774
Epoch 8 | Avg Loss: 0.3498
Epoch 9 | Avg Loss: 0.3116
Epoch 10 | Avg Loss: 0.2774
Epoch 11 | Avg Loss: 0.2431
Epoch 12 | Avg Loss: 0.2096
Epoch 13 | Avg Loss: 0.1689
Epoch 14 | Avg Loss: 0.1322
Epoch 15 | Avg Loss: 0.1049
Epoch 16 | Avg Loss: 0.0969
Epoch 17 | Avg Loss: 0.0940
Epoch 18 | Avg Loss: 0.0661
Epoch 19 | Avg Loss: 0.0605
Epoch 20 | Avg Loss: 0.0511


In [10]:
torch.save(model.state_dict(), "mmcto_phaseI_CTOD.pth")

In [10]:
print(f"\nEvaluating on Phase {PHASE} Valid Set...\n")
model.eval()

all_preds = {"fused": [], "target": []}
modalities = model.final_input_parts
for mod in modalities:
    all_preds[mod] = []

with torch.no_grad():
    for batch in test_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)

        fused_preds = out["metric_dict"].get("preds")
        target = batch["label"]

        if fused_preds is not None:
            all_preds["fused"].append(fused_preds.cpu().numpy())
            all_preds["target"].append(target.cpu().numpy())

        for mod in modalities:
            mod_pred = out["metric_dict"].get(mod)
            if mod_pred is not None:
                all_preds[mod].append(mod_pred.cpu().numpy())

for key in all_preds:
    if all_preds[key]:
        all_preds[key] = np.concatenate(all_preds[key])
    else:
        all_preds[key] = np.array([])


Evaluating on Phase I Valid Set...



C:\Users\Carol\anaconda3\envs\newthesis\Lib\site-packages\torch\nn\modules\transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(


In [11]:
def compute_metrics(preds, targets):
    preds_binary = (preds > 0.5).astype(int)
    pr = average_precision_score(targets, preds)
    roc = roc_auc_score(targets, preds)
    f1 = f1_score(targets, preds_binary)
    return pr * 100, f1 * 100, roc * 100

print(f"{'Modality':<25} {'PR':>6} {'F1':>6} {'ROC':>6}")
print("=" * 45)

for mod in modalities:
    if mod in all_preds and all_preds[mod].size:
        pr, f1, roc = compute_metrics(all_preds[mod], all_preds["target"])
        print(f"{mod:<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

pr, f1, roc = compute_metrics(all_preds["fused"], all_preds["target"])
print(f"{'All (Fused)':<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

Modality                      PR     F1    ROC
table                      77.40  67.41  42.98
summarization              73.68  51.06  36.61
smiles                     78.89   2.00  50.66
description                79.28  74.18  47.59
criteria                   78.53  39.55  51.66
enrollment                 64.03   3.64  20.52
diseases                   73.49  55.52  38.19
drugs                      79.15  58.02  50.69
All (Fused)                91.58  86.01  77.52


In [2]:
import torch
data = torch.load(r"C:\Users\Carol\Documents\Data Code and Deliverables\Data Group Part\Data\clinical-trial-outcome-prediction\Processed\ctod\phase_I_train_tokenized.pt")

for key in ["admet_metabolism", "admet_excretion", "admet_absorption", "admet_distribution", "admet_toxicity"]:
    if key in data:
        print(f"Using {key}: {data[key]}")

C:\Users\Carol\AppData\Local\Temp\ipykernel_5040\1985650478.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(r"C:\Users\Carol\Documents\Data Code and De

In [4]:
sample = data[0] if len(data) > 0 else None
if sample:
    print(f"Sample 0 keys: {sample.keys()}")  # Print the keys for the first sample
    for key in ["admet_metabolism", "admet_excretion", "admet_absorption", "admet_distribution", "admet_toxicity"]:
        if key in sample:
            print(f"Using {key}: {sample[key]}")
        else:
            print(f"{key} is not present in Sample 0.")
else:
    print("Data is empty or not loaded correctly.")

Sample 0 keys: dict_keys(['idx', 'label', 'criteria', 'enrollment', 'smiles', 'smiles_concat', 'smiles_summarization', 'smiles_transformer', 'smiles_transformer_concat', 'smiles_transformer_summarization', 'drugs', 'drugs_concat', 'drugs_summarization', 'diseases', 'diseases_concat', 'diseases_summarization', 'table', 'summarization', 'description'])
admet_metabolism is not present in Sample 0.
admet_excretion is not present in Sample 0.
admet_absorption is not present in Sample 0.
admet_distribution is not present in Sample 0.
admet_toxicity is not present in Sample 0.


In [ ]:
data = torch.load("/path/to/phase_I_train_tokenized.pt")

### PHASE II

In [9]:
print(f"\nTraining MMCTO on Phase {PHASE} with ADMET features...\n")
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for batch in train_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)
        loss = out["loss_dict"]["loss"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")


Training MMCTO on Phase II with ADMET features...

Epoch 1 | Avg Loss: 0.5760
Epoch 2 | Avg Loss: 0.5351
Epoch 3 | Avg Loss: 0.5328
Epoch 4 | Avg Loss: 0.5249
Epoch 5 | Avg Loss: 0.5129
Epoch 6 | Avg Loss: 0.4936
Epoch 7 | Avg Loss: 0.4615
Epoch 8 | Avg Loss: 0.4191
Epoch 9 | Avg Loss: 0.3651
Epoch 10 | Avg Loss: 0.3155
Epoch 11 | Avg Loss: 0.2336
Epoch 12 | Avg Loss: 0.1770
Epoch 13 | Avg Loss: 0.1279
Epoch 14 | Avg Loss: 0.0990
Epoch 15 | Avg Loss: 0.0745
Epoch 16 | Avg Loss: 0.0429
Epoch 17 | Avg Loss: 0.0456
Epoch 18 | Avg Loss: 0.0352
Epoch 19 | Avg Loss: 0.0478
Epoch 20 | Avg Loss: 0.0289


In [10]:
torch.save(model.state_dict(), "mmcto_phaseII_CTOD.pth")

In [11]:
print(f"\nEvaluating on Phase {PHASE} Valid Set...\n")
model.eval()

all_preds = {"fused": [], "target": []}
modalities = model.final_input_parts
for mod in modalities:
    all_preds[mod] = []

with torch.no_grad():
    for batch in test_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)

        fused_preds = out["metric_dict"].get("preds")
        target = batch["label"]

        if fused_preds is not None:
            all_preds["fused"].append(fused_preds.cpu().numpy())
            all_preds["target"].append(target.cpu().numpy())

        for mod in modalities:
            mod_pred = out["metric_dict"].get(mod)
            if mod_pred is not None:
                all_preds[mod].append(mod_pred.cpu().numpy())

for key in all_preds:
    if all_preds[key]:
        all_preds[key] = np.concatenate(all_preds[key])
    else:
        all_preds[key] = np.array([])


Evaluating on Phase II Valid Set...



C:\Users\Carol\anaconda3\envs\newthesis\Lib\site-packages\torch\nn\modules\transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(


In [12]:
def compute_metrics(preds, targets):
    preds_binary = (preds > 0.5).astype(int)
    pr = average_precision_score(targets, preds)
    roc = roc_auc_score(targets, preds)
    f1 = f1_score(targets, preds_binary)
    return pr * 100, f1 * 100, roc * 100

print(f"{'Modality':<25} {'PR':>6} {'F1':>6} {'ROC':>6}")
print("=" * 45)

for mod in modalities:
    if mod in all_preds:
        pr, f1, roc = compute_metrics(all_preds[mod], all_preds["target"])
        print(f"{mod:<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

pr, f1, roc = compute_metrics(all_preds["fused"], all_preds["target"])
print(f"{'All (Fused)':<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

Modality                      PR     F1    ROC
table                      76.20  53.63  46.11
summarization              76.42  77.64  47.23
smiles                     78.95   0.00  50.84
description                80.41  28.36  54.21
criteria                   77.82  25.06  49.52
enrollment                 62.20   4.89  17.77
diseases                   79.79  65.62  52.07
drugs                      79.37  47.77  52.35
All (Fused)                85.57  82.18  63.31


### PHASE III

In [10]:
print(f"\nTraining MMCTO on Phase {PHASE}...\n")
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for batch in train_loader:
        batch = move_to_device(batch, DEVICE)
        out = model(batch)
        loss = out["loss_dict"]["loss"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")


Training MMCTO on Phase III...

Epoch 1 | Avg Loss: 0.8169
Epoch 2 | Avg Loss: 0.4612
Epoch 3 | Avg Loss: 0.4253
Epoch 4 | Avg Loss: 0.4624
Epoch 5 | Avg Loss: 0.4120
Epoch 6 | Avg Loss: 0.4055
Epoch 7 | Avg Loss: 0.3653
Epoch 8 | Avg Loss: 0.3273
Epoch 9 | Avg Loss: 0.2805
Epoch 10 | Avg Loss: 0.2219
Epoch 11 | Avg Loss: 0.1731
Epoch 12 | Avg Loss: 0.1153
Epoch 13 | Avg Loss: 0.0809
Epoch 14 | Avg Loss: 0.0663
Epoch 15 | Avg Loss: 0.0524
Epoch 16 | Avg Loss: 0.0406
Epoch 17 | Avg Loss: 0.0317
Epoch 18 | Avg Loss: 0.0204
Epoch 19 | Avg Loss: 0.0144
Epoch 20 | Avg Loss: 0.0177


In [11]:
torch.save(model.state_dict(), "mmcto_phaseIII_CTOD.pth")

In [12]:
from tqdm import tqdm
import numpy as np

print(f"\nEvaluating on Phase {PHASE} Valid Set...\n")
model.eval()

# Prepare prediction containers
all_preds = {
    "fused": [],
    "target": [],
}
modalities = model.final_input_parts
for mod in modalities:
    all_preds[mod] = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating", leave=True):
        batch = move_to_device(batch, DEVICE)
        out = model(batch)

        # Store fused and target predictions
        fused_preds = out["metric_dict"].get("preds")
        target = batch["label"]

        if fused_preds is not None:
            all_preds["fused"].append(fused_preds.cpu().numpy())
            all_preds["target"].append(target.cpu().numpy())
        else:
            print("[Warning] No fused prediction returned.")

        # Store modality-specific predictions
        for mod in modalities:
            mod_pred = out["metric_dict"].get(mod)
            if mod_pred is not None:
                all_preds[mod].append(mod_pred.cpu().numpy())
            else:
                print(f"[Warning] No prediction for modality: {mod}")

# Safely concatenate arrays
for key in all_preds:
    if all_preds[key]:
        all_preds[key] = np.concatenate(all_preds[key])
    else:
        print(f"[Warning] No data to concatenate for: {key}")
        all_preds[key] = np.array([])  # or skip if not needed



Evaluating on Phase III Valid Set...



Evaluating:   0%|                                                                               | 0/33 [00:00<?, ?it/s]C:\Users\Carol\anaconda3\envs\newthesis\Lib\site-packages\torch\nn\modules\transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(
Evaluating: 100%|██████████████████████████████████████████████████████████████████████| 33/33 [00:16<00:00,  1.99it/s]


In [13]:
def compute_metrics(preds, targets):
    preds_binary = (preds > 0.5).astype(int)
    pr = average_precision_score(targets, preds)
    roc = roc_auc_score(targets, preds)
    f1 = f1_score(targets, preds_binary)
    return pr * 100, f1 * 100, roc * 100

print(f"{'Modality':<25} {'PR':>6} {'F1':>6} {'ROC':>6}")
print("=" * 45)

for mod in modalities:
    if mod in all_preds:
        pr, f1, roc = compute_metrics(all_preds[mod], all_preds["target"])
        print(f"{mod:<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

pr, f1, roc = compute_metrics(all_preds["fused"], all_preds["target"])
print(f"{'All (Fused)':<25} {pr:6.2f} {f1:6.2f} {roc:6.2f}")

Modality                      PR     F1    ROC
table                      85.89  69.53  58.41
summarization              82.87  86.99  50.20
smiles                     83.38  90.24  53.00
description                83.69   0.46  53.61
criteria                   83.20   0.46  51.47
enrollment                 68.74   0.00  20.69
diseases                   81.52  83.02  52.65
drugs                      83.20  35.27  51.85
All (Fused)                91.15  90.30  72.91
